In [1]:
import requests
import csv

# API_KEY = "08c029896c698c6fb2d287dbf0b2b15280169bd1" this is a made up key not a real one don't worry about it, there is no leek of important information here, just a placeholder for the actual key


In [ ]:
import requests
import csv
import json
import time


# ============================================================
# CONFIGURATION
# ============================================================

DOMAIN_SEARCH_LIMIT = 10


# ============================================================
# 1. FIND COMPANY DOMAIN
# ============================================================

def find_domain(company):

    url = "https://api.hunter.io/v2/domain-finder"

    params = {
        "company": company,
        "limit": 5,
        "perfect_match": "true",
        "api_key": API_KEY
    }

    response = requests.get(
        url,
        params=params
    )

    if not response.ok:

        print("\nDomain Finder error:")
        print(response.text)

        return None

    result = response.json()

    domains = result.get(
        "data",
        []
    )

    if not domains:

        print(
            f"\nNo domain found for: {company}"
        )

        return None

    # Select domain with the most emails
    best_domain = max(
        domains,
        key=lambda x: x.get(
            "email_count",
            0
        )
    )

    selected_domain = best_domain.get(
        "domain"
    )

    print("\nPossible domains:")
    print("-" * 80)

    for item in domains:

        print(
            f"{item.get('domain', ''):<35}"
            f"{item.get('email_count', 0):>10} emails"
        )

    print("-" * 80)

    print(
        f"Selected domain: {selected_domain}"
    )

    return selected_domain


# ============================================================
# 2. DOMAIN SEARCH
# ============================================================

def search_domain(domain):

    url = "https://api.hunter.io/v2/domain-search"

    params = {

        "domain": domain,

        "api_key": API_KEY,

        # Your current plan appears limited to 10
        "limit": DOMAIN_SEARCH_LIMIT,

        # Individual contacts rather than generic emails
        "type": "personal"
    }

    print(
        "\nSearching Hunter Domain Search..."
    )

    response = requests.get(
        url,
        params=params
    )

    if not response.ok:

        print(
            "\nHunter Domain Search error:"
        )

        print(
            response.text
        )

        return []

    result = response.json()

    data = result.get(
        "data",
        {}
    )

    contacts = data.get(
        "emails",
        []
    )

    print(
        f"Hunter returned "
        f"{len(contacts)} contacts."
    )

    return contacts


# ============================================================
# 3. PEOPLE ENRICHMENT
# ============================================================

def enrich_person(email):

    url = "https://api.hunter.io/v2/people/find"

    params = {
        "email": email,
        "api_key": API_KEY
    }

    response = requests.get(
        url,
        params=params
    )

    if response.status_code == 404:

        print(
            f"    No enrichment data: {email}"
        )

        return {}

    if not response.ok:

        print(
            f"    Enrichment error: {email}"
        )

        print(
            response.text
        )

        return {}

    result = response.json()

    return result.get(
        "data",
        {}
    )


# ============================================================
# 4. COMBINE DOMAIN SEARCH + ENRICHMENT
# ============================================================

def enrich_contacts(contacts):

    results = []

    total = len(contacts)

    print("\n")
    print("=" * 90)
    print("ENRICHING CONTACTS")
    print("=" * 90)

    for index, contact in enumerate(
        contacts,
        start=1
    ):

        email = contact.get(
            "value"
        )

        print(
            f"\n[{index}/{total}] {email}"
        )

        if not email:
            continue

        # --------------------------------------------
        # Hunter People Enrichment
        # --------------------------------------------

        enrichment = enrich_person(
            email
        )

        # --------------------------------------------
        # Keep EVERYTHING
        # --------------------------------------------

        combined = {}

        # Domain Search fields
        for key, value in contact.items():

            combined[
                f"domain_{key}"
            ] = value

        # Enrichment fields
        for key, value in enrichment.items():

            combined[
                f"enrichment_{key}"
            ] = value

        # --------------------------------------------
        # Convenience fields
        # --------------------------------------------

        combined["email"] = email

        combined["first_name"] = (
            contact.get("first_name")
            or enrichment.get("first_name")
        )

        combined["last_name"] = (
            contact.get("last_name")
            or enrichment.get("last_name")
        )

        combined["position"] = (
            contact.get("position")
            or enrichment.get("position")
        )

        # --------------------------------------------
        # Print location if available
        # --------------------------------------------

        location = (
            enrichment.get("location")
        )

        geo = enrichment.get(
            "geo",
            {}
        )

        print(
            f"    Location: "
            f"{location or 'Not available'}"
        )

        if isinstance(geo, dict):

            print(
                f"    City: "
                f"{geo.get('city', 'Not available')}"
            )

            print(
                f"    Country: "
                f"{geo.get('country', 'Not available')}"
            )

        results.append(
            combined
        )

        # Small delay
        time.sleep(0.1)

    return results


# ============================================================
# 5. FLATTEN NESTED JSON
# ============================================================

def flatten_dict(
    data,
    parent_key="",
    separator="."
):

    items = {}

    for key, value in data.items():

        new_key = (
            f"{parent_key}"
            f"{separator}"
            f"{key}"
            if parent_key
            else key
        )

        if isinstance(
            value,
            dict
        ):

            nested = flatten_dict(
                value,
                new_key,
                separator
            )

            items.update(
                nested
            )

        elif isinstance(
            value,
            list
        ):

            # Keep lists as JSON strings
            items[new_key] = json.dumps(
                value,
                ensure_ascii=False
            )

        else:

            items[new_key] = value

    return items


# ============================================================
# 6. SAVE COMPLETE CSV
# ============================================================

def save_csv(
    company,
    domain,
    contacts
):

    filename = (
        company
        .lower()
        .replace(" ", "_")
        .replace(".", "")
        + "_hunter_full.csv"
    )

    # Flatten every contact
    flattened = []

    for contact in contacts:

        flattened.append(
            flatten_dict(contact)
        )

    # Collect ALL fields from ALL contacts
    all_fields = set()

    for contact in flattened:

        all_fields.update(
            contact.keys()
        )

    # Put important fields first
    priority_fields = [

        "email",
        "first_name",
        "last_name",
        "position",

        "domain_value",
        "domain_first_name",
        "domain_last_name",
        "domain_position",
        "domain_department",
        "domain_seniority",
        "domain_confidence",
        "domain_linkedin",

        "enrichment_location",
        "enrichment_timeZone",

        "enrichment_geo.city",
        "enrichment_geo.state",
        "enrichment_geo.country",
        "enrichment_geo.countryCode",
        "enrichment_geo.latitude",
        "enrichment_geo.longitude"
    ]

    ordered_fields = []

    # Priority fields first
    for field in priority_fields:

        if field in all_fields:

            ordered_fields.append(
                field
            )

            all_fields.remove(
                field
            )

    # Everything else alphabetically
    ordered_fields.extend(
        sorted(all_fields)
    )

    # Add company information
    ordered_fields.insert(
        0,
        "company"
    )

    ordered_fields.insert(
        1,
        "domain"
    )

    # Write CSV
    with open(
        filename,
        "w",
        newline="",
        encoding="utf-8"
    ) as file:

        writer = csv.DictWriter(
            file,
            fieldnames=ordered_fields,
            extrasaction="ignore"
        )

        writer.writeheader()

        for contact in flattened:

            row = contact.copy()

            row["company"] = company
            row["domain"] = domain

            writer.writerow(
                row
            )

    print("\n")
    print("=" * 90)
    print("CSV SAVED")
    print("=" * 90)

    print(filename)


# ============================================================
# 7. SAVE RAW JSON
# ============================================================

def save_json(
    company,
    domain,
    contacts
):

    filename = (
        company
        .lower()
        .replace(" ", "_")
        .replace(".", "")
        + "_hunter_raw.json"
    )

    output = {

        "company": company,

        "domain": domain,

        "contacts": contacts
    }

    with open(
        filename,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            output,
            file,
            indent=4,
            ensure_ascii=False
        )

    print(
        f"Raw JSON saved: {filename}"
    )


# ============================================================
# 8. DISPLAY SUMMARY
# ============================================================

def display_summary(
    contacts
):

    print("\n")
    print("=" * 90)
    print("CONTACT SUMMARY")
    print("=" * 90)

    for index, contact in enumerate(
        contacts,
        start=1
    ):

        name = (
            f"{contact.get('first_name', '')} "
            f"{contact.get('last_name', '')}"
        ).strip()

        email = contact.get(
            "email",
            ""
        )

        position = contact.get(
            "position",
            ""
        )

        print(
            f"\n{index}. {name}"
        )

        print(
            f"   Email:    {email}"
        )

        print(
            f"   Position: {position}"
        )

        location = contact.get(
            "enrichment_location"
        )

        if location:

            print(
                f"   Location: {location}"
            )

        geo = contact.get(
            "enrichment_geo",
            {}
        )

        if isinstance(
            geo,
            dict
        ):

            print(
                f"   City:     "
                f"{geo.get('city', '')}"
            )

            print(
                f"   Country:  "
                f"{geo.get('country', '')}"
            )


# ============================================================
# 9. MAIN
# ============================================================

def main():

    print("=" * 90)
    print(
        "HUNTER FULL CONTACT + ENRICHMENT EXPORT"
    )
    print("=" * 90)

    company = input(
        "\nEnter company name: "
    ).strip()

    if not company:

        print(
            "Company name cannot be empty."
        )

        return

    # --------------------------------------------------------
    # STEP 1
    # --------------------------------------------------------

    print(
        "\n[1/4] Finding domain..."
    )

    domain = find_domain(
        company
    )

    if not domain:
        return

    # --------------------------------------------------------
    # STEP 2
    # --------------------------------------------------------

    print(
        "\n[2/4] Searching contacts..."
    )

    contacts = search_domain(
        domain
    )

    if not contacts:

        print(
            "\nNo contacts found."
        )

        return

    # --------------------------------------------------------
    # STEP 3
    # --------------------------------------------------------

    print(
        "\n[3/4] Enriching ALL contacts..."
    )

    enriched_contacts = enrich_contacts(
        contacts
    )

    # --------------------------------------------------------
    # STEP 4
    # --------------------------------------------------------

    print(
        "\n[4/4] Saving EVERYTHING..."
    )

    display_summary(
        enriched_contacts
    )

    save_csv(
        company,
        domain,
        enriched_contacts
    )

    save_json(
        company,
        domain,
        enriched_contacts
    )

    print("\nFinished.")


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    main()